# Audio Preprocessing Standalone Notebook

Notebook ini menjalankan preprocessing audio mentah secara tunggal tanpa modul eksternal selain paket Python standar.
Tujuannya adalah mengubah input audio mentah menjadi file audio yang sudah siap untuk analisis atau digunakan pada tahap inference: sample rate 16 kHz, denoise, bandpass filter, dan normalisasi.

## Install dependency
Jalankan cell ini sekali jika paket belum tersedia di environment notebook Anda.

In [ ]:
import sys
import subprocess
import pkgutil

required = [
    'librosa',
    'soundfile',
    'noisereduce',
    'matplotlib',
    'scipy'
]
installed = {pkg.name for pkg in pkgutil.iter_modules()}
missing = [pkg for pkg in required if pkg not in installed]

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('All required packages are already installed.')

## Import library dan definisi fungsi preprocessing
Semua fungsi disimpan di sini agar notebook tetap mandiri.

In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
from scipy.signal import butter, sosfilt
import matplotlib.pyplot as plt
from IPython.display import Audio, display

TARGET_SR = 16000  # Hz, standar untuk speech recognition
AUDIO_FORMAT = 'wav'

# Bandpass filter configuration untuk suara manusia
BANDPASS_LOWCUT = 80  # Hz
BANDPASS_HIGHCUT = 3000  # Hz
BANDPASS_ORDER = 4

# Noise reduction config
DENOISE_PROP_DECREASE = 0.8
DENOISE_STATIONARY = True

# Normalisasi RMS
TARGET_RMS = 0.1


def load_audio(path, sr=TARGET_SR):
    if not os.path.exists(path):
        raise FileNotFoundError(f'File tidak ditemukan: {path}')
    y, orig_sr = librosa.load(path, sr=sr, mono=True)
    return y, sr

def butter_bandpass(lowcut, highcut, fs, order=BANDPASS_ORDER):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    sos = butter(order, [low, high], btype='band', output='sos')
    return sos

def apply_bandpass(y, sr, lowcut=BANDPASS_LOWCUT, highcut=BANDPASS_HIGHCUT, order=BANDPASS_ORDER):
    sos = butter_bandpass(lowcut, highcut, sr, order)
    return sosfilt(sos, y)

def denoise_audio(y, sr, stationary=DENOISE_STATIONARY, prop_decrease=DENOISE_PROP_DECREASE):
    if len(y) < sr * 0.5:
        stationary = False
    return nr.reduce_noise(y=y, sr=sr, stationary=stationary, prop_decrease=prop_decrease)

def normalize_rms(y, target_rms=TARGET_RMS):
    rms = np.sqrt(np.mean(y**2))
    if rms <= 0:
        return y
    return y * (target_rms / rms)

def save_audio(path, y, sr):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    sf.write(path, y, sr)
    return path

def preprocess_audio_file(input_path, output_path=None):
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = f'{base}_preprocessed.{AUDIO_FORMAT}'

    print(f'Loading audio: {input_path}')
    y, sr = load_audio(input_path, sr=TARGET_SR)
    print(f'  - Duration: {len(y)/sr:.2f} s | Sample rate: {sr} Hz')

    print('Denoising...')
    y_denoised = denoise_audio(y, sr)

    print('Applying bandpass filter...')
    y_filtered = apply_bandpass(y_denoised, sr)

    print('Normalizing audio volume...')
    y_normalized = normalize_rms(y_filtered)

    print(f'Saving preprocessed file: {output_path}')
    save_audio(output_path, y_normalized, sr)
    print('Selesai.')
    return output_path, y, y_normalized, sr

def plot_waveforms(original, processed, sr):
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    times = np.arange(len(original)) / sr
    axes[0].plot(times, original, color='gray', linewidth=0.6)
    axes[0].set_title('Waveform Asli')
    axes[0].set_ylabel('Amplitude')
    axes[1].plot(times, processed, color='blue', linewidth=0.6)
    axes[1].set_title('Waveform Setelah Preprocessing')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_xlabel('Waktu (detik)')
    plt.tight_layout()
    plt.show()

## Jalankan preprocessing pada file audio mentah
Ganti `input_path` dengan path file audio Anda. Output akan disimpan sebagai WAV 16 kHz yang sudah dibersihkan.

In [ ]:
input_path = 'original_audio.wav'  # Ganti dengan path file Anda
output_path = 'preprocessed_audio.wav'  # Atau tetapkan nama file output sendiri seperti 'raw_audio_preprocessed.wav'

if not os.path.exists(input_path):
    raise FileNotFoundError(f'Ganti input_path dengan path file audio yang valid: {input_path}')

output_path, original_wave, processed_wave, sr = preprocess_audio_file(input_path, output_path)

print(f'Output tersimpan di: {output_path}')
print(f'Ukuran file output: {os.path.getsize(output_path) / 1024:.1f} KB')

plot_waveforms(original_wave, processed_wave, sr)

display(Audio(original_wave, rate=sr))
print('Audio asli di atas.')
display(Audio(processed_wave, rate=sr))
print('Audio hasil preprocessing di atas.')